In [1]:
print("hello")

hello


In [11]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

iris = load_iris()
X = iris.data[:, [0]]
y = iris.data[:, 2]    

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Sample predictions:", pred[:5])

Sample predictions: [4.6444703  4.45199739 4.06705157 2.52726828 3.87457866]


In [12]:
import joblib

In [13]:
joblib.dump(model, "model.joblib")

['model.joblib']

In [14]:
# import boto3

# s3 = boto3.client('s3')
# s3.upload_file("model.joblib", "1203tbucket", "model/model.joblib")

In [15]:
!tar -czf model.tar.gz model.joblib

In [16]:
!tar -tzf model.tar.gz

model.joblib


In [17]:
!ls

inference.py  lost+found  model.joblib	model.tar.gz  temp.ipynb


In [18]:
import boto3

s3 = boto3.client("s3")

s3.upload_file(
    "model.tar.gz",
    "1203tbucket",
    "model/model.tar.gz"
)

In [19]:
from sagemaker.sklearn.model import SKLearnModel
from sagemaker import get_execution_role

role = get_execution_role()

model = SKLearnModel(
    model_data="s3://1203tbucket/model/model.tar.gz",
    role=role,
    entry_point="inference.py",
    framework_version="1.2-1"
)

predictor = model.deploy(
    instance_type="ml.m5.large",
    initial_instance_count=1
)

------!

In [21]:
import boto3
import json

runtime = boto3.client("sagemaker-runtime")

response = runtime.invoke_endpoint(
    EndpointName=predictor.endpoint_name,
    ContentType="application/json",
    Body=json.dumps([[5.1]])
)

result = response["Body"].read().decode()

print(result)

[2.3347953679781224]


In [25]:
predictor.delete_endpoint()